In [1]:
import os
from openai import AsyncOpenAI 
from agents import Agent, Runner 
from agents import set_default_openai_client
from agents import OpenAIResponsesModel 
from agents import set_tracing_disabled
set_tracing_disabled(True)#for jupyter, remove before moving to CLI
from pandas import read_csv

import numpy as np


In [2]:
OLLAMA_BASE_URL = 'http://localhost:11434/v1'
OLLAMA_API_KEY  = 'ollama'          
OLLAMA_MODEL    = 'gpt-oss:20b'

client = AsyncOpenAI(
    api_key=OLLAMA_API_KEY,
    base_url=OLLAMA_BASE_URL,  # key detail: route requests to Ollama (local or cloud)
)

set_default_openai_client(client)

model = OpenAIResponsesModel(
    model=OLLAMA_MODEL,
    openai_client=client,
)

agent = Agent(
    name="Exploration Assistant",
    instructions="""
    You are a patient advocate. 
    You will be given a claim with relevant information and your job is to suggest a recommendation for how to best proceed.
    This recommendation can be one of the following options and nothing else: pursue, do_not_pursue, or needs_info.

    Afterwards, justify your recommendation.
    """,
    model=model,
)

#agent to assign tags from claim for playbook
tag_agent = Agent(
    name="Tag Assigning Agent",
    instructions="""
    You will be given an insurance claim and asked to assign any number of tags to it.
    If you find any of the following in the claim, it MUST be one of your tags: CO-16, CO-27, CO-29, CO-45, CO-50, CO-97
    Also assign any of the following tags if it appropriate: coding_bundling, eligibility, general, medical_necessity, missing_info, other, timely_filing, underpayment
    """,
    model=model,
)



In [3]:
claim = read_csv('data/claims.csv').iloc[0]
print(claim)



claim_id                                                    C-CO45-001
payer                                                          PayerCo
plan_type                                                        ERISA
denial_code                                                      CO-45
denial_text          Paid per contract. Allowable applied. Charge e...
days_since_denial                                                 12.0
billed_amount                                                  10000.0
paid_amount                                                     7200.0
cpt_codes                                                        99223
icd10_codes                                                        I10
prior_appeals                                                      0.0
Name: 0, dtype: object


In [4]:
"""
Loads the play‑book JSON, builds indices, and offers helper methods.
"""

import json
import os
from pathlib import Path
from typing import Dict, List, Any


class Playbook:
    def __init__(self, jsonl_path: str, docs_root: str = "docs/playbook"):
        self.jsonl_path = Path(jsonl_path)
        self.docs_root = Path(docs_root)

        # --- Load & parse JSONL --------------------------------------------
        self.chunks: List[Dict[str, Any]] = []

        with open(self.jsonl_path, "r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue  # skip blank lines
                try:
                    chunk = json.loads(line)
                except json.JSONDecodeError as exc:
                    raise ValueError(
                        f"Malformed JSON on line {line_no} of {self.jsonl_path}"
                    ) from exc
                self.chunks.append(chunk)

        # Build quick look‑ups
        self.by_id: Dict[str, Dict[str, Any]] = {c["chunk_id"]: c for c in self.chunks}
        self.tag_index: Dict[str, List[str]] = {}
        for c in self.chunks:
            for tag in c["tags"]:
                self.tag_index.setdefault(tag, []).append(c["chunk_id"])

    # ------------------------------------------------------------
    #  API helpers
    # ------------------------------------------------------------
    def get_chunks_by_tags(self, tags: List[str]) -> List[Dict[str, Any]]:
        """Return the intersection of chunks that contain *all* supplied tags."""
        if not tags:
            return []

        # Start with the set for the first tag
        common = set(self.tag_index.get(tags[0], []))
        for t in tags[1:]:
            common &= set(self.tag_index.get(t, []))

        return [self.by_id[cid] for cid in common]

    def load_markdown_for_chunk(self, chunk: Dict[str, Any]) -> str:
        """
        The `doc_id` field points at the markdown file name *without* extension.
        e.g.  `doc_id="medical_necessity.md"` → load `docs/playbook/medical_necessity.md`
        """
        doc_file = self.docs_root / f"{chunk['doc_id']}"
        if not doc_file.exists():
            return f"[ERROR: missing {doc_file}]"

        return doc_file.read_text(encoding="utf-8")

    def load_markdown_for_tags(self, tags: List[str]) -> Dict[str, str]:
        """Return a mapping of doc_id → file contents for all chunks matching the tags."""
        chunks = self.get_chunks_by_tags(tags)
        return {c["doc_id"]: self.load_markdown_for_chunk(c) for c in chunks}


claim_playbook = Playbook('data/playbook_chunks.jsonl')
# claim_playbook.load_markdown_for_tags(['missing_info','CO-29'])


In [5]:

#use playbook chunks instead of playbook

#implement tools asap

#load claims as a dataframe, parse one row at a time
claims = read_csv('data/claims.csv')
for i,row in claims.iterrows():
    claim_dict = row.to_dict()
    tag_prompt = str(row.to_dict())
    tags = await Runner.run(tag_agent, tag_prompt) 
    try:
        claim_dict['denial_code_context'] = claim_playbook.load_markdown_for_tags(list(tags))
    except:
        print('Denial code',claim_dict['denial_code'],'not found')
        claim_dict['denial_code_context'] = ''
    prompt = "Here is the claim information: " + str(claim_dict)
    print('Prompt:\n',prompt)
    print('-'*100)
    result = await Runner.run(agent, prompt) 
    print(result.final_output)
    break

Denial code CO-45 not found
Prompt:
 Here is the claim information: {'claim_id': 'C-CO45-001', 'payer': 'PayerCo', 'plan_type': 'ERISA', 'denial_code': 'CO-45', 'denial_text': 'Paid per contract. Allowable applied. Charge exceeds fee schedule/maximum allowable.', 'days_since_denial': 12.0, 'billed_amount': 10000.0, 'paid_amount': 7200.0, 'cpt_codes': '99223', 'icd10_codes': 'I10', 'prior_appeals': 0.0, 'denial_code_context': ''}
----------------------------------------------------------------------------------------------------
needs_info

The denial code CO‑45 indicates that the payer believes the billed amount ($10,000) exceeds the fee schedule or maximum allowable amount for CPT 99223. While the claim was partially paid ($7,200), the large discrepancy ($2,800) suggests either an error in the billing or that the provider may lack documentation to justify the higher charge. Before deciding to appeal or dismiss the claim, additional information is required:

1. The payer’s specific fee

In [6]:
import tiktoken

# 20B GPT‑OSS model (same as "gpt-3.5-turbo" for tokenization)
enc = tiktoken.get_encoding("cl100k_base")   # use the model name

tokens = enc.encode(prompt)
print(f"Prompt length: {len(tokens)} tokens")



Prompt length: 135 tokens


In [7]:
#get unique tags once
chunks: List[Dict[str, Any]] = []
jsonl_path = 'data/playbook_chunks.jsonl'
with open(jsonl_path, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue  # skip blank lines
        try:
            chunk = json.loads(line)
        except json.JSONDecodeError as exc:
            raise ValueError(
                f"Malformed JSON on line {line_no} of {jsonl_path}"
            ) from exc
        chunks.append(chunk)
np.unique([x for cnk in chunks for x in cnk['tags']])


array(['CO-16', 'CO-27', 'CO-29', 'CO-45', 'CO-50', 'CO-97',
       'coding_bundling', 'eligibility', 'general', 'medical_necessity',
       'missing_info', 'other', 'timely_filing', 'underpayment'],
      dtype='<U17')

In [8]:
model = OpenAIResponsesModel(
    model=OLLAMA_MODEL,
    openai_client=client,
)



In [9]:
dir(result),result.to_state()

(['__abstractmethods__',
  '__annotations__',
  '__class__',
  '__dataclass_fields__',
  '__dataclass_params__',
  '__del__',
  '__delattr__',
  '__dict__',
  '__dir__',
  '__doc__',
  '__eq__',
  '__format__',
  '__ge__',
  '__get_pydantic_core_schema__',
  '__getattribute__',
  '__gt__',
  '__hash__',
  '__init__',
  '__init_subclass__',
  '__le__',
  '__lt__',
  '__module__',
  '__ne__',
  '__new__',
  '__post_init__',
  '__reduce__',
  '__reduce_ex__',
  '__repr__',
  '__setattr__',
  '__sizeof__',
  '__slots__',
  '__str__',
  '__subclasshook__',
  '__weakref__',
  '_abc_impl',
  '_auto_previous_response_id',
  '_conversation_id',
  '_current_turn',
  '_current_turn_persisted_item_count',
  '_last_agent',
  '_last_agent_ref',
  '_last_processed_response',
  '_model_input_items',
  '_original_input',
  '_previous_response_id',
  '_release_last_agent_reference',
  '_tool_use_tracker_snapshot',
  '_trace_state',
  'context_wrapper',
  'final_output',
  'final_output_as',
  'input',
 

In [10]:
row

claim_id                                                    C-CO45-001
payer                                                          PayerCo
plan_type                                                        ERISA
denial_code                                                      CO-45
denial_text          Paid per contract. Allowable applied. Charge e...
days_since_denial                                                 12.0
billed_amount                                                  10000.0
paid_amount                                                     7200.0
cpt_codes                                                        99223
icd10_codes                                                        I10
prior_appeals                                                      0.0
Name: 0, dtype: object